# Responses API

Agent SDK uses the Responses API internally, so it is good to get a sense of how to use it directly. My learning so far has been to use the Chat Completion API because it is similar to other non-OpenAI offerings as well. But now it is time to learn a subset of the Responses API.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import pydantic as pt
import json
from utils import LLM

In [2]:
load_dotenv()

True

In [3]:
client = OpenAI()

## Structured Outputs
The biggest difference between Chat and Responses, is that Responses does not support Pydantic objects when generating structured outputs. I have to put in a JSON schema. I can still use Pydantic's `model_json_schema` and `model_validate_json` to serialize/deserialize the JSON to objects.

Another point of difference is that instead of the more intuitively named `response_format` input parameter, I have to use `text` parameter to pass in the output json schema.

In [4]:
class CalendarEvent(pt.BaseModel):
    name: str
    date: str
    participants: list[str]


CalendarEvent.model_json_schema()

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object'}

In [5]:
schema = CalendarEvent.model_json_schema()
schema.update({"additionalProperties": False})
schema

{'properties': {'name': {'title': 'Name', 'type': 'string'},
  'date': {'title': 'Date', 'type': 'string'},
  'participants': {'items': {'type': 'string'},
   'title': 'Participants',
   'type': 'array'}},
 'required': ['name', 'date', 'participants'],
 'title': 'CalendarEvent',
 'type': 'object',
 'additionalProperties': False}

In [6]:
text = {
    "format": {
        "type": "json_schema",
        "name": "CalendarEvent",
        "schema": schema,
        "strict": True,
    }
}

In the below example copied from the documentation, the input is using the Chat API style input with different roles and contents. In subsequent examples in this notebook I'll try to use the new Responses API style input with `instructions` and `input` attribute.

In [7]:
resp = client.responses.create(
    model=LLM.PRE_FAST_MINI,
    input=[
        {"role": "system", "content": "Extract the event information."},
        {
            "role": "user",
            "content": "Alice and Bob are going to science fair Tuesday next week.",
        },
    ],
    text=text,  # type: ignore
)
resp

Response(id='resp_68052bcb10148191be5c69de48b61ca500b5e3148471fde3', created_at=1745169355.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_68052bcb83b881918bd60c3c4a81923300b5e3148471fde3', content=[ResponseOutputText(annotations=[], text='{"name":"Science Fair","date":"2023-10-03","participants":["Alice","Bob"]}', type='output_text')], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, max_output_tokens=None, previous_response_id=None, reasoning=Reasoning(effort=None, generate_summary=None, summary=None), status='completed', text=ResponseTextConfig(format=ResponseFormatTextJSONSchemaConfig(schema_={'properties': {'name': {'title': 'Name', 'type': 'string'}, 'date': {'title': 'Date', 'type': 'string'}, 'participants': {'items': {'type': 'string'}, 'title': 'Participants', 'type': 'array'

In [8]:
resp.output_text

'{"name":"Science Fair","date":"2023-10-03","participants":["Alice","Bob"]}'

In [9]:
calendar_event = CalendarEvent.model_validate_json(resp.output_text)
calendar_event

CalendarEvent(name='Science Fair', date='2023-10-03', participants=['Alice', 'Bob'])

Some more examples.

In [10]:
class Cookie(pt.BaseModel):
    flavor: str
    calories: int | None

In [11]:
schema = Cookie.model_json_schema()
schema.update({"additionalProperties": False})

In [12]:
text = {
    "format": {
        "type": "json_schema",
        "name": "Cookie",
        "schema": schema,
        "strict": True,
    }
}

In [13]:
resp = client.responses.create(
    model=LLM.PRE_FAST_MINI,
    instructions="Extract cookie information about the cookie that AP likes.",
    input="AP loves to eat Oatmeal Raisin cookies! At 180 calories, these are lower than the Chocolate Chip cookies that Anika likes.",
    text=text,  # type: ignore
)

In [14]:
cookie = Cookie.model_validate_json(resp.output_text)
cookie

Cookie(flavor='Oatmeal Raisin', calories=180)

In [15]:
class Step(pt.BaseModel):
    explanation: str
    output: str


class Answer(pt.BaseModel):
    steps: list[Step]
    final_answer: str


Answer.model_json_schema()

{'$defs': {'Step': {'properties': {'explanation': {'title': 'Explanation',
     'type': 'string'},
    'output': {'title': 'Output', 'type': 'string'}},
   'required': ['explanation', 'output'],
   'title': 'Step',
   'type': 'object'}},
 'properties': {'steps': {'items': {'$ref': '#/$defs/Step'},
   'title': 'Steps',
   'type': 'array'},
  'final_answer': {'title': 'Final Answer', 'type': 'string'}},
 'required': ['steps', 'final_answer'],
 'title': 'Answer',
 'type': 'object'}

In [16]:
schema = Answer.model_json_schema()
schema["additionalProperties"] = False
schema["$defs"]["Step"]["additionalProperties"] = False

In [17]:
text = {
    "format": {
        "type": "json_schema",
        "name": "Answer",
        "schema": schema,
        "strict": True,
    }
}

In [21]:
resp = client.responses.create(
    # model=LLM.PRE_FAST_MINI,  This also seems to work reasonably well
    model=LLM.RES_MINI,
    # reasoning={"effort": "medium"},  This is the default
    instructions="You are a helpful math tutor. Guide the user through the solution step-by-step.",
    input="How can I solve 8x + 7 = 23?",
    text=text,  # type: ignore
)
resp

Response(id='resp_68052c24b6a48191beed11d62d97a43e0f663b7235cf165f', created_at=1745169444.0, error=None, incomplete_details=None, instructions='You are a helpful math tutor. Guide the user through the solution step-by-step.', metadata={}, model='o4-mini-2025-04-16', object='response', output=[ResponseReasoningItem(id='rs_68052c250e5c8191bbf2748a815fa2a90f663b7235cf165f', summary=[], type='reasoning', status=None), ResponseOutputMessage(id='msg_68052c27b6588191a02f34c1236c8efe0f663b7235cf165f', content=[ResponseOutputText(annotations=[], text='{"steps":[{"explanation":"Start with the equation 8x + 7 = 23. To isolate the term with x, subtract 7 from both sides.","output":"8x = 23 − 7 → 8x = 16"},{"explanation":"Now divide both sides by 8 to solve for x.","output":"x = 16 ÷ 8 → x = 2"}],"final_answer":"x = 2"}', type='output_text')], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, max_output_token

In [22]:
answer = Answer.model_validate_json(resp.output_text)
answer

Answer(steps=[Step(explanation='Start with the equation 8x + 7 = 23. To isolate the term with x, subtract 7 from both sides.', output='8x = 23 − 7 → 8x = 16'), Step(explanation='Now divide both sides by 8 to solve for x.', output='x = 16 ÷ 8 → x = 2')], final_answer='x = 2')

In [23]:
print(answer.final_answer)
for step in answer.steps:
    print("---")
    print(step.explanation)
    print(step.output)

x = 2
---
Start with the equation 8x + 7 = 23. To isolate the term with x, subtract 7 from both sides.
8x = 23 − 7 → 8x = 16
---
Now divide both sides by 8 to solve for x.
x = 16 ÷ 8 → x = 2


## Functions



In [24]:
%reset -f

In [25]:
from dotenv import load_dotenv
from openai import OpenAI, pydantic_function_tool
import pydantic as pt
import requests
import json
from utils import LLM, ResponseFunctionToolCall
from typing import cast, Any

In [26]:
load_dotenv()

True

In [27]:
client = OpenAI()

In [28]:
class GetWeatherRequest(pt.BaseModel):
    """
    Get the current temperature for provided coordinates in celsius.
    """

    latitude: float = pt.Field(description="Latitude of the location.")
    longitude: float = pt.Field(description="Longitude of the location.")

In [29]:
def get_weather(latitude: float, longitude: float) -> int:
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]["temperature_2m"]

In [31]:
class SendEMailRequest(pt.BaseModel):
    """Send an email to a given recipient with a subject and message."""

    to: str = pt.Field(description="The recipient's email address.")
    subject: str = pt.Field(description="Email subject line.")
    body: str = pt.Field(description="Body of the email message.")

In [32]:
class SendEmailResponse(pt.BaseModel):
    """SMTP response to send mail"""

    code: int
    status: str

In [33]:
def send_email(to: str, subject: str, body: str) -> SendEmailResponse:
    return SendEmailResponse(code=250, status="OK")

In [34]:
weather_tool: dict[str, Any] = cast(
    dict, pydantic_function_tool(GetWeatherRequest)["function"]
)
weather_tool["type"] = "function"

email_tool: dict[str, Any] = cast(
    dict, pydantic_function_tool(SendEMailRequest)["function"]
)
email_tool["type"] = "function"

tools = [weather_tool, email_tool]

In [35]:
prompt = {
    "role": "user",
    "content": "Send an email to Avilay telling him about the weather in Kolkata. His email address is avilay@gmail.com.",
}

In [36]:
resp_1 = client.responses.create(model=LLM.PRE_FAST_MINI, input=[prompt], tools=tools)  # type: ignore
resp_1

Response(id='resp_68052c51ccac8191b2eac403239e40e60a1b4ae4f4f24bd1', created_at=1745169489.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFunctionToolCall(arguments='{"latitude":22.5726,"longitude":88.3639}', call_id='call_yhnmJwJjHnVwn9t0lVwI5XDK', name='GetWeatherRequest', type='function_call', id='fc_68052c5268c481919f6b01480a4d104b0a1b4ae4f4f24bd1', status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='GetWeatherRequest', parameters={'description': 'Get the current temperature for provided coordinates in celsius.', 'properties': {'latitude': {'description': 'Latitude of the location.', 'title': 'Latitude', 'type': 'number'}, 'longitude': {'description': 'Longitude of the location.', 'title': 'Longitude', 'type': 'number'}}, 'required': ['latitude', 'longitude'], 'title': 'GetWeatherRequest', 'type': 'object', 'additionalProperties': Fa

In [37]:
resp_1.output

[ResponseFunctionToolCall(arguments='{"latitude":22.5726,"longitude":88.3639}', call_id='call_yhnmJwJjHnVwn9t0lVwI5XDK', name='GetWeatherRequest', type='function_call', id='fc_68052c5268c481919f6b01480a4d104b0a1b4ae4f4f24bd1', status='completed')]

In [38]:
resp_1.output_text

''

In [39]:
tool_call = cast(ResponseFunctionToolCall, resp_1.output[0])
kwargs = json.loads(tool_call.arguments)
kwargs

{'latitude': 22.5726, 'longitude': 88.3639}

In [40]:
ret = get_weather(**kwargs)
ret

28.8

In [41]:
input_: list[ResponseFunctionToolCall | dict[str, str]] = [
    prompt
]  # The original user prompt
input_.append(tool_call)
assert tool_call.id
input_.append(
    {"type": "function_call_output", "call_id": tool_call.call_id, "output": str(ret)}
)
input_

[{'role': 'user',
  'content': 'Send an email to Avilay telling him about the weather in Kolkata. His email address is avilay@gmail.com.'},
 ResponseFunctionToolCall(arguments='{"latitude":22.5726,"longitude":88.3639}', call_id='call_yhnmJwJjHnVwn9t0lVwI5XDK', name='GetWeatherRequest', type='function_call', id='fc_68052c5268c481919f6b01480a4d104b0a1b4ae4f4f24bd1', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_yhnmJwJjHnVwn9t0lVwI5XDK',
  'output': '28.8'}]

In [42]:
resp_2 = client.responses.create(model=LLM.PRE_FAST_MINI, input=input_, tools=tools)  # type: ignore
resp_2

Response(id='resp_68052c6e3cb881919a4fffa0e7aa73120a1b4ae4f4f24bd1', created_at=1745169518.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFunctionToolCall(arguments='{"to":"avilay@gmail.com","subject":"Weather Update for Kolkata","body":"Hello Avilay,\\n\\nThe current temperature in Kolkata is 28.8°C.\\n\\nBest regards,"}', call_id='call_4B6MCmttDJ2TUTvJxczgP4It', name='SendEMailRequest', type='function_call', id='fc_68052c7240ac81918514b5c3e5a653b90a1b4ae4f4f24bd1', status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='GetWeatherRequest', parameters={'description': 'Get the current temperature for provided coordinates in celsius.', 'properties': {'latitude': {'description': 'Latitude of the location.', 'title': 'Latitude', 'type': 'number'}, 'longitude': {'description': 'Longitude of the location.', 'title': 'Longitude', 'type': 'number'}

In [43]:
resp_2.output

[ResponseFunctionToolCall(arguments='{"to":"avilay@gmail.com","subject":"Weather Update for Kolkata","body":"Hello Avilay,\\n\\nThe current temperature in Kolkata is 28.8°C.\\n\\nBest regards,"}', call_id='call_4B6MCmttDJ2TUTvJxczgP4It', name='SendEMailRequest', type='function_call', id='fc_68052c7240ac81918514b5c3e5a653b90a1b4ae4f4f24bd1', status='completed')]

In [44]:
tool_call = cast(ResponseFunctionToolCall, resp_2.output[0])
kwargs = json.loads(tool_call.arguments)
kwargs

{'to': 'avilay@gmail.com',
 'subject': 'Weather Update for Kolkata',
 'body': 'Hello Avilay,\n\nThe current temperature in Kolkata is 28.8°C.\n\nBest regards,'}

In [45]:
ret = send_email(**kwargs)
ret.model_dump_json()

'{"code":250,"status":"OK"}'

In [46]:
input_.append(tool_call)
assert tool_call.id
input_.append(
    {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(ret.model_dump_json()),
    }
)
input_

[{'role': 'user',
  'content': 'Send an email to Avilay telling him about the weather in Kolkata. His email address is avilay@gmail.com.'},
 ResponseFunctionToolCall(arguments='{"latitude":22.5726,"longitude":88.3639}', call_id='call_yhnmJwJjHnVwn9t0lVwI5XDK', name='GetWeatherRequest', type='function_call', id='fc_68052c5268c481919f6b01480a4d104b0a1b4ae4f4f24bd1', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_yhnmJwJjHnVwn9t0lVwI5XDK',
  'output': '28.8'},
 ResponseFunctionToolCall(arguments='{"to":"avilay@gmail.com","subject":"Weather Update for Kolkata","body":"Hello Avilay,\\n\\nThe current temperature in Kolkata is 28.8°C.\\n\\nBest regards,"}', call_id='call_4B6MCmttDJ2TUTvJxczgP4It', name='SendEMailRequest', type='function_call', id='fc_68052c7240ac81918514b5c3e5a653b90a1b4ae4f4f24bd1', status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_4B6MCmttDJ2TUTvJxczgP4It',
  'output': '"{\\"code\\":250,\\"status\\":\\"OK\\"}"'}]

In [47]:
resp_3 = client.responses.create(model=LLM.PRE_FAST_MINI, input=input_, tools=tools)  # type: ignore
resp_3

Response(id='resp_68052c7c8d54819184e8d34b6c4ce4ba0a1b4ae4f4f24bd1', created_at=1745169532.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseOutputMessage(id='msg_68052c7d27d88191a8a14f91417810d30a1b4ae4f4f24bd1', content=[ResponseOutputText(annotations=[], text='I have sent an email to Avilay about the current weather in Kolkata. If you need anything else, let me know!', type='output_text')], role='assistant', status='completed', type='message')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='GetWeatherRequest', parameters={'description': 'Get the current temperature for provided coordinates in celsius.', 'properties': {'latitude': {'description': 'Latitude of the location.', 'title': 'Latitude', 'type': 'number'}, 'longitude': {'description': 'Longitude of the location.', 'title': 'Longitude', 'type': 'number'}}, 'required': ['latitude', 'longitude'], 't

In [48]:
resp_3.output_text

'I have sent an email to Avilay about the current weather in Kolkata. If you need anything else, let me know!'